In [ ]:
"""
HIPPA Risk Assessment Automation Tool
Analyzes hospital system architecture for HIPPA compliance gaps
"""

import json
from datetime import datetime
from typing import Dict, List, Tuple
from dataclasses import dataclass, field
from enum import Enum
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak
from reportlab.lib.enums import TA_CENTER, TA_LEFT


class RiskLevel(Enum):
    CRITICAL = 4
    HIGH = 3
    MEDIUM = 2
    LOW = 1


class NISTBaseline(Enum):
    LOW = 1
    MODERATE = 2
    HIGH = 3


@dataclass
class Risk:
    category: str
    risk_id: str
    description: str
    likelihood: int  # 1-5
    impact: int  # 1-5
    risk_score: int = 0
    risk_level: RiskLevel = RiskLevel.LOW
    hippa_requirements: List[str] = field(default_factory=list)
    nist_controls: List[str] = field(default_factory=list)
    remediation: str = ""
    
    def __post_init__(self):
        self.risk_score = self.likelihood * self.impact
        if self.risk_score >= 20:
            self.risk_level = RiskLevel.CRITICAL
        elif self.risk_score >= 12:
            self.risk_level = RiskLevel.HIGH
        elif self.risk_score >= 6:
            self.risk_level = RiskLevel.MEDIUM
        else:
            self.risk_level = RiskLevel.LOW


class HIPPARiskAssessment:
    
    # HIPPA Security Rule mapping
    HIPPA_CONTROLS = {
        "Access Control": {
            "164.312(a)(1)": "Implement technical policies and procedures for electronic information systems that maintain ePHI to allow access only to those persons or software programs that have been granted access rights",
            "164.312(a)(2)(i)": "Assign a unique name and/or number for identifying and tracking user identity",
            "164.312(a)(2)(ii)": "Establish (and implement as needed) procedures for obtaining necessary ePHI during an emergency",
            "164.312(a)(2)(iii)": "Implement procedures that terminate an electronic session after a predetermined time of inactivity",
            "164.312(a)(2)(iv)": "Implement a mechanism to encrypt and decrypt ePHI"
        },
        "Audit Controls": {
            "164.312(b)": "Implement hardware, software, and/or procedural mechanisms that record and examine activity in information systems that contain or use ePHI"
        },
        "Integrity": {
            "164.312(c)(1)": "Implement policies and procedures to protect ePHI from improper alteration or destruction",
            "164.312(c)(2)": "Implement electronic mechanisms to corroborate that ePHI has not been altered or destroyed in an unauthorized manner"
        },
        "Transmission Security": {
            "164.312(e)(1)": "Implement technical security measures to guard against unauthorized access to ePHI that is being transmitted over an electronic communications network",
            "164.312(e)(2)(i)": "Implement a mechanism to encrypt ePHI whenever deemed appropriate",
            "164.312(e)(2)(ii)": "Implement security measures to ensure that electronically transmitted ePHI is not improperly modified without detection until disposed of"
        }
    }
    
    # NIST 800-53 Control mappings
    NIST_CONTROLS = {
        "Access Control": ["AC-1", "AC-2", "AC-3", "AC-6", "AC-7", "AC-17", "AC-19", "AC-20"],
        "Audit Controls": ["AU-1", "AU-2", "AU-3", "AU-6", "AU-9", "AU-12"],
        "Integrity": ["SI-1", "SI-7", "SC-8", "SC-13"],
        "Transmission Security": ["SC-8", "SC-13", "SC-23"]
    }
    
    def __init__(self):
        self.risks: List[Risk] = []
        self.system_data = {}
        
    def load_system_architecture(self, json_path: str) -> Dict:
        """Load hospital system architecture from JSON file"""
        with open(json_path, 'r') as f:
            self.system_data = json.load(f)
        return self.system_data
    
    def interactive_assessment(self) -> Dict:
        """Interactive questionnaire to gather system architecture information"""
        print("\n" + "="*70)
        print("HIPPA RISK ASSESSMENT - INTERACTIVE QUESTIONNAIRE")
        print("="*70)
        print("\nPlease answer the following questions about your hospital system.")
        print("Answer with 'yes'/'no' or provide specific values as requested.\n")
        
        system_data = {}
        
        # Authentication Section
        print("\n--- AUTHENTICATION & ACCESS CONTROL ---")
        system_data['authentication'] = {
            'mfa_enabled': self._ask_yes_no("Is multi-factor authentication (MFA) enabled for ePHI access?"),
            'password_policy': input("What is your password policy? (simple/complex/very_complex): ").strip().lower() or "simple"
        }
        
        system_data['access_control'] = {
            'role_based': self._ask_yes_no("Is role-based access control (RBAC) implemented?"),
            'least_privilege': self._ask_yes_no("Are least privilege principles enforced?")
        }
        
        # Session Management
        print("\n--- SESSION MANAGEMENT ---")
        timeout = input("What is the session timeout in minutes? (0 for no timeout): ").strip()
        system_data['session_management'] = {
            'timeout_minutes': int(timeout) if timeout.isdigit() else 0
        }
        
        # Encryption
        print("\n--- ENCRYPTION ---")
        system_data['encryption'] = {
            'data_at_rest': self._ask_yes_no("Is ePHI encrypted at rest?"),
            'algorithm': None
        }
        if system_data['encryption']['data_at_rest']:
            system_data['encryption']['algorithm'] = input("What encryption algorithm is used? (e.g., AES-256): ").strip() or "Unknown"
        
        # Logging & Audit Controls
        print("\n--- LOGGING & AUDIT CONTROLS ---")
        system_data['logging'] = {
            'enabled': self._ask_yes_no("Is audit logging enabled for ePHI access?"),
            'retention_days': 0,
            'monitoring_enabled': False,
            'tamper_protection': False
        }
        
        if system_data['logging']['enabled']:
            retention = input("How many days are audit logs retained? ").strip()
            system_data['logging']['retention_days'] = int(retention) if retention.isdigit() else 0
            system_data['logging']['monitoring_enabled'] = self._ask_yes_no("Is real-time log monitoring enabled?")
            system_data['logging']['tamper_protection'] = self._ask_yes_no("Are logs protected against tampering?")
        
        # Integrity Controls
        print("\n--- INTEGRITY CONTROLS ---")
        system_data['integrity_controls'] = {
            'hash_verification': self._ask_yes_no("Is hash verification used to detect ePHI alterations?"),
            'version_control': self._ask_yes_no("Is version control implemented for ePHI?")
        }
        
        # Backup
        print("\n--- BACKUP & RECOVERY ---")
        system_data['backup'] = {
            'enabled': self._ask_yes_no("Are automated backups of ePHI enabled?"),
            'frequency': 'none',
            'encryption': False
        }
        
        if system_data['backup']['enabled']:
            system_data['backup']['frequency'] = input("Backup frequency? (daily/weekly/monthly): ").strip().lower() or "none"
            system_data['backup']['encryption'] = self._ask_yes_no("Are backups encrypted?")
        
        # Transmission Security
        print("\n--- TRANSMISSION SECURITY ---")
        system_data['transmission'] = {
            'tls_enabled': self._ask_yes_no("Is TLS/SSL enabled for ePHI transmission?"),
            'tls_version': 'none',
            'integrity_check': False
        }
        
        if system_data['transmission']['tls_enabled']:
            system_data['transmission']['tls_version'] = input("What TLS version is used? (e.g., TLS1.2, TLS1.3): ").strip() or "TLS1.0"
            system_data['transmission']['integrity_check'] = self._ask_yes_no("Are integrity checks (MAC/signatures) used for transmissions?")
        
        # Remote Access
        print("\n--- REMOTE ACCESS ---")
        system_data['remote_access'] = {
            'enabled': self._ask_yes_no("Is remote access to ePHI systems enabled?"),
            'vpn_required': False
        }
        
        if system_data['remote_access']['enabled']:
            system_data['remote_access']['vpn_required'] = self._ask_yes_no("Is VPN required for remote access?")
        
        # Organization Info
        print("\n--- ORGANIZATION INFORMATION ---")
        system_data['organization'] = input("Organization/Hospital name: ").strip() or "Unknown Organization"
        
        self.system_data = system_data
        
        print("\n" + "="*70)
        print("✓ Questionnaire complete! Data collected.")
        print("="*70)
        
        return system_data
    
    def _ask_yes_no(self, question: str) -> bool:
        """Helper method to ask yes/no questions"""
        while True:
            answer = input(f"{question} (yes/no): ").strip().lower()
            if answer in ['yes', 'y']:
                return True
            elif answer in ['no', 'n']:
                return False
            else:
                print("Please answer 'yes' or 'no'")
    
    def save_system_data(self, filepath: str = "system_architecture.json"):
        """Save collected system data to JSON file"""
        with open(filepath, 'w') as f:
            json.dump(self.system_data, f, indent=2)
        print(f"\n✓ System architecture saved to: {filepath}")
    
    def assess_access_control_risks(self) -> List[Risk]:
        """Identify Access Control risks"""
        risks = []
        
        # Check for authentication mechanisms
        auth = self.system_data.get('authentication', {})
        if not auth.get('mfa_enabled', False):
            risks.append(Risk(
                category="Access Control",
                risk_id="AC-001",
                description="Multi-factor authentication not enabled for ePHI access",
                likelihood=4,
                impact=5,
                hippa_requirements=["164.312(a)(2)(i)"],
                nist_controls=["AC-2", "AC-7"],
                remediation="Implement MFA for all users accessing ePHI systems"
            ))
        
        # Check session management
        session = self.system_data.get('session_management', {})
        if session.get('timeout_minutes', 0) > 15 or session.get('timeout_minutes', 0) == 0:
            risks.append(Risk(
                category="Access Control",
                risk_id="AC-002",
                description="Session timeout not configured or exceeds recommended threshold",
                likelihood=3,
                impact=3,
                hippa_requirements=["164.312(a)(2)(iii)"],
                nist_controls=["AC-12"],
                remediation="Configure automatic session timeout to 15 minutes or less of inactivity"
            ))
        
        # Check encryption at rest
        encryption = self.system_data.get('encryption', {})
        if not encryption.get('data_at_rest', False):
            risks.append(Risk(
                category="Access Control",
                risk_id="AC-003",
                description="ePHI not encrypted at rest",
                likelihood=4,
                impact=5,
                hippa_requirements=["164.312(a)(2)(iv)"],
                nist_controls=["SC-13", "SC-28"],
                remediation="Implement AES-256 encryption for all ePHI storage systems"
            ))
        
        # Check role-based access control
        rbac = self.system_data.get('access_control', {})
        if not rbac.get('role_based', False):
            risks.append(Risk(
                category="Access Control",
                risk_id="AC-004",
                description="Role-based access control not implemented",
                likelihood=4,
                impact=4,
                hippa_requirements=["164.312(a)(1)"],
                nist_controls=["AC-3", "AC-6"],
                remediation="Implement least privilege access using role-based access controls"
            ))
        
        return risks
    
    def assess_audit_control_risks(self) -> List[Risk]:
        """Identify Audit Control risks"""
        risks = []
        
        logging = self.system_data.get('logging', {})
        
        # Check audit logging
        if not logging.get('enabled', False):
            risks.append(Risk(
                category="Audit Controls",
                risk_id="AU-001",
                description="Audit logging not enabled for ePHI access",
                likelihood=5,
                impact=4,
                hippa_requirements=["164.312(b)"],
                nist_controls=["AU-2", "AU-3", "AU-12"],
                remediation="Enable comprehensive audit logging for all ePHI access and modifications"
            ))
        
        # Check log retention
        if logging.get('retention_days', 0) < 2190:  # 6 years
            risks.append(Risk(
                category="Audit Controls",
                risk_id="AU-002",
                description="Audit log retention period insufficient",
                likelihood=3,
                impact=3,
                hippa_requirements=["164.312(b)"],
                nist_controls=["AU-11"],
                remediation="Configure audit log retention for minimum 6 years as per HIPAA requirements"
            ))
        
        # Check log monitoring
        if not logging.get('monitoring_enabled', False):
            risks.append(Risk(
                category="Audit Controls",
                risk_id="AU-003",
                description="Real-time audit log monitoring not implemented",
                likelihood=4,
                impact=4,
                hippa_requirements=["164.312(b)"],
                nist_controls=["AU-6"],
                remediation="Implement SIEM solution with real-time monitoring and alerting"
            ))
        
        # Check log integrity
        if not logging.get('tamper_protection', False):
            risks.append(Risk(
                category="Audit Controls",
                risk_id="AU-004",
                description="Audit logs not protected against tampering",
                likelihood=3,
                impact=4,
                hippa_requirements=["164.312(b)"],
                nist_controls=["AU-9"],
                remediation="Implement write-once log storage or cryptographic log signing"
            ))
        
        return risks
    
    def assess_integrity_risks(self) -> List[Risk]:
        """Identify Integrity risks"""
        risks = []
        
        integrity = self.system_data.get('integrity_controls', {})
        
        # Check data integrity mechanisms
        if not integrity.get('hash_verification', False):
            risks.append(Risk(
                category="Integrity",
                risk_id="IN-001",
                description="No mechanism to verify ePHI has not been altered",
                likelihood=3,
                impact=5,
                hippa_requirements=["164.312(c)(2)"],
                nist_controls=["SI-7", "SC-8"],
                remediation="Implement cryptographic hashing or digital signatures for ePHI"
            ))
        
        # Check backup and recovery
        backup = self.system_data.get('backup', {})
        if not backup.get('enabled', False):
            risks.append(Risk(
                category="Integrity",
                risk_id="IN-002",
                description="ePHI backup procedures not implemented",
                likelihood=4,
                impact=5,
                hippa_requirements=["164.312(c)(1)"],
                nist_controls=["CP-9"],
                remediation="Implement automated encrypted backups with tested recovery procedures"
            ))
        
        # Check version control
        if not integrity.get('version_control', False):
            risks.append(Risk(
                category="Integrity",
                risk_id="IN-003",
                description="No versioning system for ePHI modifications",
                likelihood=2,
                impact=3,
                hippa_requirements=["164.312(c)(1)"],
                nist_controls=["SI-7"],
                remediation="Implement version control system to track ePHI modifications"
            ))
        
        return risks
    
    def assess_transmission_security_risks(self) -> List[Risk]:
        """Identify Transmission Security risks"""
        risks = []
        
        transmission = self.system_data.get('transmission', {})
        
        # Check encryption in transit
        if not transmission.get('tls_enabled', False):
            risks.append(Risk(
                category="Transmission Security",
                risk_id="TR-001",
                description="ePHI transmitted without encryption",
                likelihood=5,
                impact=5,
                hippa_requirements=["164.312(e)(2)(i)"],
                nist_controls=["SC-8", "SC-13"],
                remediation="Implement TLS 1.2 or higher for all ePHI transmissions"
            ))
        
        # Check TLS version
        if transmission.get('tls_version', '') < 'TLS1.2':
            risks.append(Risk(
                category="Transmission Security",
                risk_id="TR-002",
                description="Outdated TLS version in use",
                likelihood=4,
                impact=4,
                hippa_requirements=["164.312(e)(1)"],
                nist_controls=["SC-8"],
                remediation="Upgrade to TLS 1.2 or TLS 1.3 and disable older protocols"
            ))
        
        # Check transmission integrity
        if not transmission.get('integrity_check', False):
            risks.append(Risk(
                category="Transmission Security",
                risk_id="TR-003",
                description="No mechanism to detect transmission tampering",
                likelihood=3,
                impact=4,
                hippa_requirements=["164.312(e)(2)(ii)"],
                nist_controls=["SC-8"],
                remediation="Implement message authentication codes (MAC) or digital signatures"
            ))
        
        # Check VPN for remote access
        remote = self.system_data.get('remote_access', {})
        if remote.get('enabled', False) and not remote.get('vpn_required', False):
            risks.append(Risk(
                category="Transmission Security",
                risk_id="TR-004",
                description="Remote access to ePHI without VPN",
                likelihood=4,
                impact=4,
                hippa_requirements=["164.312(e)(1)"],
                nist_controls=["AC-17", "SC-8"],
                remediation="Require VPN with strong encryption for all remote ePHI access"
            ))
        
        return risks
    
    def perform_assessment(self) -> List[Risk]:
        """Perform comprehensive risk assessment"""
        self.risks = []
        self.risks.extend(self.assess_access_control_risks())
        self.risks.extend(self.assess_audit_control_risks())
        self.risks.extend(self.assess_integrity_risks())
        self.risks.extend(self.assess_transmission_security_risks())
        return self.risks
    
    def determine_nist_baseline(self) -> NISTBaseline:
        """Determine appropriate NIST 800-53 baseline based on risk profile"""
        critical_count = sum(1 for r in self.risks if r.risk_level == RiskLevel.CRITICAL)
        high_count = sum(1 for r in self.risks if r.risk_level == RiskLevel.HIGH)
        
        if critical_count >= 3 or high_count >= 5:
            return NISTBaseline.HIGH
        elif critical_count >= 1 or high_count >= 2:
            return NISTBaseline.MODERATE
        else:
            return NISTBaseline.LOW
    
    def generate_pdf_report(self, output_path: str = "hippa_risk_assessment.pdf"):
        """Generate comprehensive PDF report"""
        doc = SimpleDocTemplate(output_path, pagesize=letter)
        story = []
        styles = getSampleStyleSheet()
        
        # Custom styles
        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=24,
            textColor=colors.HexColor('#1a1a1a'),
            spaceAfter=30,
            alignment=TA_CENTER
        )
        
        heading_style = ParagraphStyle(
            'CustomHeading',
            parent=styles['Heading2'],
            fontSize=16,
            textColor=colors.HexColor('#2c5aa0'),
            spaceAfter=12,
            spaceBefore=12
        )
        
        # Title
        story.append(Paragraph("HIPPA Security Rule Risk Assessment Report", title_style))
        story.append(Spacer(1, 0.2*inch))
        
        # Executive Summary
        story.append(Paragraph("Executive Summary", heading_style))
        
        baseline = self.determine_nist_baseline()
        summary_data = [
            ["Assessment Date:", datetime.now().strftime("%B %d, %Y")],
            ["Total Risks Identified:", str(len(self.risks))],
            ["Critical Risks:", str(sum(1 for r in self.risks if r.risk_level == RiskLevel.CRITICAL))],
            ["High Risks:", str(sum(1 for r in self.risks if r.risk_level == RiskLevel.HIGH))],
            ["Medium Risks:", str(sum(1 for r in self.risks if r.risk_level == RiskLevel.MEDIUM))],
            ["Low Risks:", str(sum(1 for r in self.risks if r.risk_level == RiskLevel.LOW))],
            ["Recommended NIST Baseline:", baseline.name]
        ]
        
        summary_table = Table(summary_data, colWidths=[2.5*inch, 3.5*inch])
        summary_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (0, -1), colors.HexColor('#e8f4f8')),
            ('TEXTCOLOR', (0, 0), (-1, -1), colors.black),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 10),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
            ('LEFTPADDING', (0, 0), (-1, -1), 8),
            ('RIGHTPADDING', (0, 0), (-1, -1), 8),
            ('TOPPADDING', (0, 0), (-1, -1), 6),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
        ]))
        
        story.append(summary_table)
        story.append(Spacer(1, 0.3*inch))
        
        # Detailed Findings by Category
        story.append(PageBreak())
        story.append(Paragraph("Detailed Risk Findings", heading_style))
        
        categories = ["Access Control", "Audit Controls", "Integrity", "Transmission Security"]
        
        for category in categories:
            category_risks = [r for r in self.risks if r.category == category]
            
            if category_risks:
                story.append(Spacer(1, 0.2*inch))
                story.append(Paragraph(f"{category} ({len(category_risks)} risks)", 
                                     ParagraphStyle('CategoryHeading', parent=styles['Heading3'], 
                                                  textColor=colors.HexColor('#2c5aa0'))))
                
                for risk in sorted(category_risks, key=lambda x: x.risk_score, reverse=True):
                    # Risk details table
                    risk_data = [
                        ["Risk ID:", risk.risk_id],
                        ["Risk Level:", risk.risk_level.name],
                        ["Risk Score:", f"{risk.risk_score} (Likelihood: {risk.likelihood}, Impact: {risk.impact})"],
                        ["Description:", risk.description],
                        ["HIPPA Requirements:", ", ".join(risk.hippa_requirements)],
                        ["NIST Controls:", ", ".join(risk.nist_controls)],
                        ["Remediation:", risk.remediation]
                    ]
                    
                    risk_table = Table(risk_data, colWidths=[1.5*inch, 5*inch])
                    
                    # Color coding based on risk level
                    if risk.risk_level == RiskLevel.CRITICAL:
                        bg_color = colors.HexColor('#ffcccc')
                    elif risk.risk_level == RiskLevel.HIGH:
                        bg_color = colors.HexColor('#ffe6cc')
                    elif risk.risk_level == RiskLevel.MEDIUM:
                        bg_color = colors.HexColor('#ffffcc')
                    else:
                        bg_color = colors.HexColor('#e6f3ff')
                    
                    risk_table.setStyle(TableStyle([
                        ('BACKGROUND', (0, 1), (0, 1), bg_color),
                        ('BACKGROUND', (1, 1), (1, 1), bg_color),
                        ('BACKGROUND', (0, 0), (0, 0), colors.HexColor('#f0f0f0')),
                        ('TEXTCOLOR', (0, 0), (-1, -1), colors.black),
                        ('ALIGN', (0, 0), (0, -1), 'LEFT'),
                        ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
                        ('FONTSIZE', (0, 0), (-1, -1), 9),
                        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
                        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                        ('LEFTPADDING', (0, 0), (-1, -1), 6),
                        ('RIGHTPADDING', (0, 0), (-1, -1), 6),
                        ('TOPPADDING', (0, 0), (-1, -1), 5),
                        ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
                    ]))
                    
                    story.append(risk_table)
                    story.append(Spacer(1, 0.15*inch))
        
        # Compliance Gap Summary
        story.append(PageBreak())
        story.append(Paragraph("HIPPA Compliance Gap Analysis", heading_style))
        
        gap_text = f"""
        Based on the assessment, {len(self.risks)} compliance gaps were identified across the HIPAA Security Rule categories.
        The recommended NIST 800-53 baseline is <b>{baseline.name}</b>, which provides appropriate security controls
        for the current risk profile.
        """
        story.append(Paragraph(gap_text, styles['BodyText']))
        story.append(Spacer(1, 0.2*inch))
        
        # Priority Remediation Actions
        story.append(Paragraph("Priority Remediation Actions", heading_style))
        
        critical_and_high = [r for r in self.risks if r.risk_level in [RiskLevel.CRITICAL, RiskLevel.HIGH]]
        critical_and_high.sort(key=lambda x: x.risk_score, reverse=True)
        
        priority_data = [["Priority", "Risk ID", "Category", "Remediation Action"]]
        
        for idx, risk in enumerate(critical_and_high[:10], 1):
            priority_data.append([
                str(idx),
                risk.risk_id,
                risk.category,
                risk.remediation
            ])
        
        priority_table = Table(priority_data, colWidths=[0.6*inch, 0.8*inch, 1.5*inch, 3.6*inch])
        priority_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2c5aa0')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 8),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
            ('LEFTPADDING', (0, 0), (-1, -1), 5),
            ('RIGHTPADDING', (0, 0), (-1, -1), 5),
            ('TOPPADDING', (0, 0), (-1, -1), 5),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
        ]))
        
        story.append(priority_table)
        
        # Build PDF
        doc.build(story)
        print(f"\n✓ Assessment complete! Report generated: {output_path}")
        return output_path


# Example usage
def run_interactive_mode():
    """Run assessment in interactive mode"""
    print("=" * 70)
    print("HIPPA RISK ASSESSMENT AUTOMATION TOOL")
    print("=" * 70)
    
    assessment = HIPPARiskAssessment()
    
    # Interactive questionnaire
    print("\n[1/4] Collecting system information...")
    assessment.interactive_assessment()
    
    # Optionally save the data
    save = input("\nWould you like to save this data for future use? (yes/no): ").strip().lower()
    if save in ['yes', 'y']:
        filename = input("Enter filename (default: system_architecture.json): ").strip() or "system_architecture.json"
        assessment.save_system_data(filename)
    
    # Perform assessment
    print("\n[2/4] Performing risk assessment...")
    risks = assessment.perform_assessment()
    
    print(f"\n✓ Identified {len(risks)} risks")
    print(f"  - Critical: {sum(1 for r in risks if r.risk_level == RiskLevel.CRITICAL)}")
    print(f"  - High: {sum(1 for r in risks if r.risk_level == RiskLevel.HIGH)}")
    print(f"  - Medium: {sum(1 for r in risks if r.risk_level == RiskLevel.MEDIUM)}")
    print(f"  - Low: {sum(1 for r in risks if r.risk_level == RiskLevel.LOW)}")
    
    baseline = assessment.determine_nist_baseline()
    print(f"\n✓ Recommended NIST 800-53 Baseline: {baseline.name}")
    
    # Generate report
    print("\n[3/4] Generating PDF report...")
    report_name = input("Enter report filename (default: hippa_risk_assessment.pdf): ").strip() or "hipaa_risk_assessment.pdf"
    assessment.generate_pdf_report(report_name)
    
    print("\n[4/4] Displaying risk summary...")
    print("\nTop 5 Critical/High Risks:")
    critical_high = [r for r in risks if r.risk_level in [RiskLevel.CRITICAL, RiskLevel.HIGH]]
    critical_high.sort(key=lambda x: x.risk_score, reverse=True)
    
    for i, risk in enumerate(critical_high[:5], 1):
        print(f"\n  {i}. [{risk.risk_level.name}] {risk.risk_id}: {risk.description}")
        print(f"     Remediation: {risk.remediation}")
    
    print("\n" + "=" * 70)
    print("Assessment Complete!")
    print("=" * 70)


def run_json_mode():
    """Run assessment using existing JSON file"""
    print("=" * 70)
    print("HIPPA RISK ASSESSMENT AUTOMATION TOOL - JSON Mode")
    print("=" * 70)
    
    json_file = input("\nEnter path to system architecture JSON file: ").strip()
    
    assessment = HIPPARiskAssessment()
    
    try:
        print("\n[1/3] Loading system architecture...")
        assessment.load_system_architecture(json_file)
        
        print("[2/3] Performing risk assessment...")
        risks = assessment.perform_assessment()
        
        print(f"\n✓ Identified {len(risks)} risks")
        print(f"  - Critical: {sum(1 for r in risks if r.risk_level == RiskLevel.CRITICAL)}")
        print(f"  - High: {sum(1 for r in risks if r.risk_level == RiskLevel.HIGH)}")
        print(f"  - Medium: {sum(1 for r in risks if r.risk_level == RiskLevel.MEDIUM)}")
        print(f"  - Low: {sum(1 for r in risks if r.risk_level == RiskLevel.LOW)}")
        
        baseline = assessment.determine_nist_baseline()
        print(f"\n✓ Recommended NIST 800-53 Baseline: {baseline.name}")
        
        print("\n[3/3] Generating PDF report...")
        assessment.generate_pdf_report()
        
        print("\n" + "=" * 70)
        print("Assessment Complete!")
        print("=" * 70)
        
    except FileNotFoundError:
        print(f"\n✗ Error: File '{json_file}' not found.")
    except json.JSONDecodeError:
        print(f"\n✗ Error: Invalid JSON format in '{json_file}'.")


if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("HIPPA RISK ASSESSMENT AUTOMATION TOOL")
    print("=" * 70)
    print("\nChoose assessment mode:")
    print("1. Interactive Mode (Answer questions)")
    print("2. JSON Mode (Load from existing file)")
    print("3. Demo Mode (Use sample data)")
    
    choice = input("\nEnter choice (1/2/3): ").strip()
    
    if choice == "1":
        run_interactive_mode()
    elif choice == "2":
        run_json_mode()
    elif choice == "3":
        # Create sample JSON for demo
        sample_system = {
            "organization": "Sample Hospital System",
            "authentication": {
                "mfa_enabled": False,
                "password_policy": "complex"
            },
            "session_management": {
                "timeout_minutes": 30
            },
            "encryption": {
                "data_at_rest": False,
                "algorithm": None
            },
            "access_control": {
                "role_based": True,
                "least_privilege": False
            },
            "logging": {
                "enabled": True,
                "retention_days": 365,
                "monitoring_enabled": False,
                "tamper_protection": False
            },
            "integrity_controls": {
                "hash_verification": False,
                "version_control": True
            },
            "backup": {
                "enabled": True,
                "frequency": "daily",
                "encryption": False
            },
            "transmission": {
                "tls_enabled": True,
                "tls_version": "TLS1.1",
                "integrity_check": False
            },
            "remote_access": {
                "enabled": True,
                "vpn_required": False
            }
        }
        
        with open('demo_hospital_system.json', 'w') as f:
            json.dump(sample_system, f, indent=2)
        
        print("\n✓ Demo data created: demo_hospital_system.json")
        
        assessment = HIPPARiskAssessment()
        print("\n[1/3] Loading demo system architecture...")
        assessment.load_system_architecture('demo_hospital_system.json')
        
        print("[2/3] Performing risk assessment...")
        risks = assessment.perform_assessment()
        
        print(f"\n✓ Identified {len(risks)} risks")
        print(f"  - Critical: {sum(1 for r in risks if r.risk_level == RiskLevel.CRITICAL)}")
        print(f"  - High: {sum(1 for r in risks if r.risk_level == RiskLevel.HIGH)}")
        print(f"  - Medium: {sum(1 for r in risks if r.risk_level == RiskLevel.MEDIUM)}")
        print(f"  - Low: {sum(1 for r in risks if r.risk_level == RiskLevel.LOW)}")
        
        baseline = assessment.determine_nist_baseline()
        print(f"\n✓ Recommended NIST 800-53 Baseline: {baseline.name}")
        
        print("\n[3/3] Generating PDF report...")
        assessment.generate_pdf_report()
        
        print("\n" + "=" * 70)
        print("Assessment Complete!")
        print("=" * 70)
    else:
        print("\n✗ Invalid choice. Please run the program again.")


HIPPA RISK ASSESSMENT AUTOMATION TOOL

Choose assessment mode:
1. Interactive Mode (Answer questions)
2. JSON Mode (Load from existing file)
3. Demo Mode (Use sample data)



Enter choice (1/2/3):  1


HIPPA RISK ASSESSMENT AUTOMATION TOOL

[1/4] Collecting system information...

HIPPA RISK ASSESSMENT - INTERACTIVE QUESTIONNAIRE

Please answer the following questions about your hospital system.
Answer with 'yes'/'no' or provide specific values as requested.


--- AUTHENTICATION & ACCESS CONTROL ---


Is multi-factor authentication (MFA) enabled for ePHI access? (yes/no):  yes
